## **Prueba de ingeniería de datos**

Este notebook corresponde a una prueba de ingeniería de datos, en la cual se proporcionaron archivos CSV para implementar un proceso de carga, almacenamiento y procesamiento de datos mediante un **pipeline de micro-batches**.

## **Herramientas y servicios utilizados**

- **Pandas:** utilizado para la lectura y procesamiento de los archivos CSV. Se eligió debido a que los archivos proporcionados manejan una cantidad de datos pequeña 31 registros el de mayor tamaño.
- **MySQL:** utilizado como base de datos para almacenar las transacciones y las estadísticas acumuladas.
- **Railway:** utilizado para desplegar la base de datos MySQL en la nube y disponer de almacenamiento remoto. Debido a la inactividad ya que es una instancia gratuita, esta puede entrar temporalmente en suspensión, al realizar una nueva conexión, se reactiva y **posteriormente funciona con normalidad**.
- **Google Colab Secrets:** utilizado para almacenar de forma privada las variables de conexión a la base de datos, como:
  - **`MYSQL_HOST`**
  - **`MYSQL_PORT`**
  - **`MYSQL_USER`**
  - **`MYSQL_PASSWORD`**
  - **`MYSQL_DATABASE`**
evitando incluirlas directamente en el código.
- **Carpeta `data`:** utilizada para almacenar los archivos CSV proporcionados para la prueba.

### 1. Instalación y importacion de librerías

In [149]:
!pip install mysql-connector-python

In [150]:
import pandas as pd
import os
import gc
import mysql.connector

try:
    from google.colab import userdata
    _EN_COLAB = True
except ImportError:
    userdata = None
    _EN_COLAB = False
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
# En Colab las credenciales salen de Secrets (userdata).
# En local salen de variables de entorno / archivo .env.

## 2. Conexion a la base de datos MySql

In [151]:
def _get_credencial(nombre):
  """
  Obtiene una credencial desde Colab Secrets o variables de entorno.

  Parameters
  ----------
  nombre : str
      Nombre de la credencial.

  Returns
  -------
  str
      Valor de la credencial.
  """
  if _EN_COLAB and userdata is not None:
      try:
          valor = userdata.get(nombre)
          if valor:
              return valor
      except Exception:
          pass
  return os.getenv(nombre)

host = _get_credencial("MYSQL_HOST")
port = int(_get_credencial("MYSQL_PORT"))
user = _get_credencial("MYSQL_USER")
password = _get_credencial("MYSQL_PASSWORD")
database = _get_credencial("MYSQL_DATABASE")

def conexion_mysql():
  """
  Crea y retorna una conexión a la base de datos MySQL.

  Returns
  -------
  MySQLConnection
      Conexión activa a MySQL.
  """
  conexion = mysql.connector.connect(
      host=host,
      port=port,
      user=user,
      password=password,
      database=database
  )

  print("Conexión exitosa a MySQL")
  return conexion

# Testear conexion
conexion = conexion_mysql()
cursor = conexion.cursor()
cursor.execute("SELECT VERSION();")
resultado = cursor.fetchone()
print("Versión de MySQL:", resultado[0])

# Cerrar conexión
cursor.close()
conexion.close()

Conexión exitosa a MySQL
Versión de MySQL: 9.4.0


## 3. Creación de esquemas del proceso

- **1. transactions:** (Almacena los datos de los .csv)
  - **id (INT):** Identificador único y incremental de cada registro
  - **timestamp (DATE):** Fecha del registro
  - **price (DECIMAL(10,2)):** Precio de la transacción
  - **user_id (INT):** Identificador del usuario

- **2. statistics:** Almacena las estadísticas acumuladas
  - **id:** Identificador del registro de estadísticas
  - **count_total** Total de registros procesados
  - **count_price** Cantidad de precios válidos
  - **sum_price** Suma acumulada de precios
  - **min_price** Precio mínimo acumulado
  - **max_price** Precio máximo acumulado

**Ejemplo:**
  - id = siempre va a ser 1 ya que se va a estar actualizando
  - count_total = 4 hay 4 registros en total.
  - count_price = 3 solo 3 tienen precio válido.
  - sum_price = 200 (50+ 80 +70)
  - avg_price = 200 / 3 = 66.67


**Decisión de tipo:**
  - `DATE` el origen viene sin hora (formato `M/D/YYYY`, ej. `1/10/2012`), por eso se usa `DATE` y no `DATETIME`/`TIMESTAMP` para no inventar un `00:00:00` que no existe en la fuente.
  - `price` permite `NULL` porque la data trae precios faltantes (4 nulos: 2 en `2012-1` y 2 en `2012-4`); no se imputan ni se borran filas porque no es un pipeline de limpieza, es de ingesta + estadísticas (`count_total` vs `count_price`).

**Nota**: origen sin hora, por eso DATE y no DATETIME.

In [152]:
def crear_tablas():
    conexion = conexion_mysql()
    cursor = conexion.cursor()

    # Tabla donde se almacenan los eventos/transacciones
    # price permite valores NULL porque la data trae precios faltantes
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS transactions (
          id INT AUTO_INCREMENT PRIMARY KEY
              COMMENT 'Identificador único de la transacción',
          timestamp DATE NOT NULL
              COMMENT 'Fecha de la transacción',
          price DECIMAL(10,2) NULL
              COMMENT 'Precio de la transacción',
          user_id INT NOT NULL
              COMMENT 'Identificador del usuario asociado a la transacción'
      )
      COMMENT = 'Registros de las transacciones procesadas por el pipeline';
    """)

    print("Tabla 'transactions' creada correctamente.")

    # Tabla para mantener las estadísticas acumuladas
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS statistics (
          id INT PRIMARY KEY
              COMMENT 'Identificador del registro de estadísticas',
          count_total BIGINT NOT NULL DEFAULT 0
              COMMENT 'Cantidad total de registros procesados',
          count_price BIGINT NOT NULL DEFAULT 0
              COMMENT 'Cantidad de registros con precio válido',
          sum_price DECIMAL(20,2) NOT NULL DEFAULT 0
              COMMENT 'Suma acumulada de los precios',
          min_price DECIMAL(10,2)
              COMMENT 'Precio mínimo acumulado',
          max_price DECIMAL(10,2)
              COMMENT 'Precio máximo acumulado'
      )
      COMMENT = 'Estadísticas acumuladas del procesamiento de los micro-batches';
    """)

    print("Tabla 'statistics' creada correctamente.")

    # Crear el registro inicial de estadísticas
    cursor.execute("""
        INSERT IGNORE INTO statistics
        (id, count_total, count_price, sum_price, min_price, max_price)
        VALUES (1, 0, 0, 0, NULL, NULL)
    """)

    print("Insert inicial en 'statistics' insertado correctamente.")

    conexion.commit()

    cursor.close()
    conexion.close()

    print("Tablas creadas correctamente.")

crear_tablas()

Conexión exitosa a MySQL
Tabla 'transactions' creada correctamente.
Tabla 'statistics' creada correctamente.
Insert inicial en 'statistics' insertado correctamente.
Tablas creadas correctamente.


## 4. Procesamientos de achivos en micro Batches

Definición de funciones y llamado de los mismas para el procesamiento de los archivos

In [153]:
def insertar_datos(datos, cursor, conexion):
    """
    Inserta los datos de un micro-batch en la tabla transactions.

    No hace commit: el commit/rollback lo hace `procesar_micro_batch`
    para que inserción + actualización de estadísticas sea atómica.

    Parameters
    ----------
    datos : list
        Registros a insertar en la base de datos (price=None -> NULL).
    cursor : MySQLCursor
        Cursor activo de MySQL.
    conexion : MySQLConnection
        Conexión activa a MySQL.

    Returns
    -------
    None
        No retorna ningún valor.
    """
    query = """
        INSERT INTO transactions (
          timestamp,
          price,
          user_id
        )
        VALUES (%s, %s, %s)
    """
    cursor.executemany(query, datos)
    print(f"{cursor.rowcount} registros insertados correctamente.")


In [154]:
def actualizar_estadisticas(
    cursor,
    conexion,
    count_total,
    count_price,
    sum_price,
    min_price,
    max_price
):
    """
    Actualiza las estadísticas acumuladas en la base de datos.

    No hace SELECT sobre transactions, solo actualización incremental.
    Si el micro-batch no trae precios válidos (min_price=None),
    solo actualiza conteos/suma para no corromper min/max con NULL.

    Parameters
    ----------
    cursor : MySQLCursor
        Cursor activo de MySQL.
    conexion : MySQLConnection
        Conexión activa a MySQL.
    count_total : int
        Cantidad de registros del micro-batch.
    count_price : int
        Cantidad de precios válidos (None -> no se cuenta).
    sum_price : float
        Suma de los precios del micro-batch.
    min_price : float | None
        Precio mínimo del micro-batch o None si no hay válidos.
    max_price : float | None
        Precio máximo del micro-batch o None si no hay válidos.

    Returns
    -------
    None
        No retorna ningún valor.
    """
    if min_price is None or max_price is None or count_price == 0:
        sql = """
        UPDATE statistics
        SET
            count_total = count_total + %s,
            count_price = count_price + %s,
            sum_price = sum_price + %s
        WHERE id = 1
        """
        cursor.execute(sql, (count_total, count_price, sum_price))
        return

    sql = """
    UPDATE statistics
    SET
        count_total = count_total + %s,
        count_price = count_price + %s,
        sum_price = sum_price + %s,
        min_price = CASE
            WHEN min_price IS NULL THEN %s
            ELSE LEAST(min_price, %s)
        END,
        max_price = CASE
            WHEN max_price IS NULL THEN %s
            ELSE GREATEST(max_price, %s)
        END
    WHERE id = 1
    """
    valores = (
        count_total,
        count_price,
        sum_price,
        min_price,
        min_price,
        max_price,
        max_price
    )
    cursor.execute(sql, valores)

In [155]:
def consultar_estadisticas(cursor):
    """
    Consulta las estadísticas acumuladas en la base de datos.

    Parameters
    ----------
    cursor : MySQLCursor
        Cursor activo de MySQL.

    Returns
    -------
    tuple
        Estadísticas acumuladas: cantidad, promedio, mínimo y máximo.
    """
    sql = """
    SELECT
        count_total,
        sum_price / NULLIF(count_price, 0) AS avg_price,
        min_price,
        max_price
    FROM statistics
    WHERE id = 1
    """
    cursor.execute(sql)

    return cursor.fetchone()

In [156]:
def procesar_micro_batch(file, cursor, conexion):
    """
    Procesa un CSV, inserta sus datos en MySQL y actualiza
    las estadísticas acumuladas de forma incremental y atómica.

    - Solo carga en memoria el archivo actual (micro-batch).
    - Preserva nulos como NULL (no imputa) para no afectar la data.
    - Hace un solo commit al final: inserción + stats.

    Parameters
    ----------
    file : str
        Nombre del archivo CSV.
    cursor : MySQLCursor
        Cursor activo de MySQL.
    conexion : MySQLConnection
        Conexión activa a MySQL.

    Returns
    -------
    None
        No retorna ningún valor.
    """
    file_path = os.path.join(data_path, file)
    print(f"\nProcesando micro-batch: {file}")

    # Lectura del CSV actual (solo este archivo en memoria)
    df = pd.read_csv(file_path)

    # Casteo de tipos (origen timestamp sin hora, DATE)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["user_id"] = pd.to_numeric(df["user_id"], downcast="integer")

    # Preservar nulos como None - NULL en MySQL (no imputar con 0)
    datos = [
        (
            fila.timestamp.date() if not pd.isna(fila.timestamp) else None,
            None if pd.isna(fila.price) else float(fila.price),
            int(fila.user_id)
        )
        for fila in df.itertuples(index=False)
    ]

    # Estadísticas del micro-batch (count excluye NaN)
    count_total = int(len(df))
    count_price = int(df["price"].count())
    sum_price = float(df["price"].sum())
    min_price = float(df["price"].min()) if count_price > 0 else None
    max_price = float(df["price"].max()) if count_price > 0 else None

    # Transacción atómica: insert + stats con un solo commit
    try:
        insertar_datos(datos, cursor, conexion)
        print("Actualizando estadisticas")
        actualizar_estadisticas(
            cursor,
            conexion,
            count_total,
            count_price,
            sum_price,
            min_price,
            max_price
        )
        conexion.commit()
        print("Micro-batch confirmado.")
    except Exception as e:
        conexion.rollback()
        print("Error en micro-batch, rollback:", e)
        raise
    finally:
        # Mostrar estadísticas acumuladas
        estadisticas = consultar_estadisticas(cursor)
        print("Estadísticas acumuladas:")
        print(f"Filas cargadas: {estadisticas[0]}")
        print(f"Precio promedio: {estadisticas[1]}")
        print(f"Precio mínimo: {estadisticas[2]}")
        print(f"Precio máximo: {estadisticas[3]}")
        # Liberar memoria (no mantener todo el dataset)
        del df
        gc.collect()


## 4.1. Definir la carpeta donde están los CSV

En esta carpeta se deben subir los datos a procesar

In [157]:
data_path = "data"

## 4.2. Obtener los archivos CSV excepto validation.csv

Busca todos los archivos **.csv** de la carpeta **data**, excepto **validation.csv**, y los ordena.

In [158]:
files = sorted([
    file for file in os.listdir(data_path)
    if file.endswith(".csv") and file != "validation.csv"
])

print("Archivos a procesar")
print(files)

Archivos a procesar
['2012-1.csv', '2012-2.csv', '2012-3.csv', '2012-4.csv', '2012-5.csv']


## 4.3 Procesar los 5 archivos y obtener sus estadisticas

- Procesa reutilizando la función **procesar_micro_batch**
- Inserta los datos y actualiza las estadisticas
- Retorna las estadisticas despues de procesar cada archivo durante su ejecución

In [159]:
conexion = conexion_mysql()
cursor = conexion.cursor()

for file in files:
    procesar_micro_batch(
        file,
        cursor,
        conexion
    )

cursor.close()
conexion.close()

print("\nProcesamiento de los archivos ejecutado correctamente.")

Conexión exitosa a MySQL

Procesando micro-batch: 2012-1.csv
22 registros insertados correctamente.
Actualizando estadisticas
Micro-batch confirmado.
Estadísticas acumuladas:
Filas cargadas: 22
Precio promedio: 59.650000
Precio mínimo: 14.00
Precio máximo: 97.00

Procesando micro-batch: 2012-2.csv
29 registros insertados correctamente.
Actualizando estadisticas
Micro-batch confirmado.
Estadísticas acumuladas:
Filas cargadas: 51
Precio promedio: 56.795918
Precio mínimo: 10.00
Precio máximo: 100.00

Procesando micro-batch: 2012-3.csv
31 registros insertados correctamente.
Actualizando estadisticas
Micro-batch confirmado.
Estadísticas acumuladas:
Filas cargadas: 82
Precio promedio: 57.912500
Precio mínimo: 10.00
Precio máximo: 100.00

Procesando micro-batch: 2012-4.csv
30 registros insertados correctamente.
Actualizando estadisticas
Micro-batch confirmado.
Estadísticas acumuladas:
Filas cargadas: 112
Precio promedio: 57.777778
Precio mínimo: 10.00
Precio máximo: 100.00

Procesando micro-b

## 4. Comprobación de resultados (punto 3 del PDF) 5 CSV

> La regla del PDF que prohíbe `SELECT AVG(price)...` aplica solo a las **actualizaciones incrementales** del pipeline (ya resuelto con tabla `statistics`) ahora hacemos una consulta de este tipo solo para la validacion de resultados despues de procesados los 5 archivos y llenado las estadisticas **consulta de comprobación de resultados**.
> - Para la **comprobación final** el PDF pide explícitamente consultar la BD después de procesar los 5 CSV, para esto se consultan las dos tablas y se valida si hay diferencias

In [160]:
def ejecutar_consulta(query, cursor, columns=[]):
  """
  Ejecuta una consulta SQL y retorna los resultados como DataFrame.

  Parameters
  ----------
  query : str
      Consulta SQL a ejecutar.
  cursor : MySQLCursor
      Cursor activo de MySQL.
  columns : list, optional
      Nombres de las columnas del DataFrame.

  Returns
  -------
  pandas.DataFrame
      Resultados de la consulta en formato DataFrame.
  """
  cursor.execute(query)
  datos = cursor.fetchall()
  df = pd.DataFrame(
      datos,
      columns=columns
  )
  return df

In [161]:
def consultar_datos():
  """
  Consulta y compara las estadísticas acumuladas.

  Calcula las estadísticas directamente desde transactions,
  las compara con la tabla statistics y genera una alerta
  cuando existen diferencias.

  Returns
  -------
  pandas.DataFrame
      DataFrame con estadísticas, diferencias y estado de la validación.
  """
  conexion = conexion_mysql()
  cursor = conexion.cursor()

  query_verificacion = """
  WITH transactions_stats AS (
      SELECT
          COUNT(*) AS total,
          AVG(price) AS promedio,
          MIN(price) AS minimo,
          MAX(price) AS maximo
      FROM transactions
  ),

  statistics_stats AS (
      SELECT
          count_total AS total,
          sum_price / NULLIF(count_price, 0) AS promedio,
          min_price AS minimo,
          max_price AS maximo
      FROM statistics
      WHERE id = 1
  )

  SELECT
      'transactions' AS fuente,
      total,
      promedio,
      minimo,
      maximo,
      'REFERENCIA' AS estado
  FROM transactions_stats

  UNION ALL

  SELECT
      'statistics' AS fuente,
      total,
      promedio,
      minimo,
      maximo,
      'ACUMULADO' AS estado
  FROM statistics_stats

  UNION ALL

  SELECT
      'diferencias' AS fuente,
      t.total - s.total AS total,
      t.promedio - s.promedio AS promedio,
      t.minimo - s.minimo AS minimo,
      t.maximo - s.maximo AS maximo,
      CASE
          WHEN t.total = s.total
          AND ABS(t.promedio - s.promedio) < 0.01
          AND t.minimo = s.minimo
          AND t.maximo = s.maximo
          THEN 'OK'
          ELSE 'ALERTA: HAY DIFERENCIAS'
      END AS estado
  FROM transactions_stats t
  CROSS JOIN statistics_stats s;
  """
  df_verificacion = ejecutar_consulta(
      query_verificacion,
      cursor,
      ["fuente", "total", "promedio", "minimo", "maximo", "estado"]
  )

  cursor.close()
  conexion.close()
  return df_verificacion

display(consultar_datos())

Conexión exitosa a MySQL


,fuente,total,promedio,minimo,maximo,estado
0,transactions,143,57.884892,10.00,100.00,REFERENCIA
1,statistics,143,57.884892,10.00,100.00,ACUMULADO
2,diferencias,0,0.000000,0.00,0.00,OK


## 4.4 Procesar el archivo validation.csv y obtener estadisticas

- Procesa reutilizando la función **procesar_micro_batch**
- Inserta los datos y actualiza las estadisticas
- Retorna las estadisticas despues de procesar el archivo

In [162]:
conexion = conexion_mysql()
cursor = conexion.cursor()

procesar_micro_batch(
    "validation.csv",
    cursor,
    conexion
)

cursor.close()
conexion.close()

print("\nProcesamiento de validation.csv terminada.")

Conexión exitosa a MySQL

Procesando micro-batch: validation.csv
8 registros insertados correctamente.
Actualizando estadisticas
Micro-batch confirmado.
Estadísticas acumuladas:
Filas cargadas: 151
Precio promedio: 57.006803
Precio mínimo: 10.00
Precio máximo: 100.00

Procesamiento de validation.csv terminada.


## 4.5 Comprobación de resultados (punto 3 del PDF) validation.csv

> Para la **comprobación final** el PDF pide consultar la DB después de procesar `validation.csv`.
- Reutilizamos la funcion **consultar_datos** y vovemos a consultar las tablas para saber si hay diferencias y ver el cambio de las estadisticas.

In [163]:
display(consultar_datos())

Conexión exitosa a MySQL


,fuente,total,promedio,minimo,maximo,estado
0,transactions,151,57.006803,10.00,100.00,REFERENCIA
1,statistics,151,57.006803,10.00,100.00,ACUMULADO
2,diferencias,0,0.000000,0.00,0.00,OK


## Anexo: Re-ejecución limpia (idempotencia)

Si se va a re-correr el notebook completo, ejecutar esta celda primero para no duplicar la informacion. No se ejecutó automáticamente para preservar la corrida evaluada.

In [164]:
def limpiar_tablas():
    """Deja el pipeline en estado inicial para una re-ejecución limpia."""
    conexion = conexion_mysql()
    cursor = conexion.cursor()
    cursor.execute("TRUNCATE TABLE transactions")
    cursor.execute("""
      UPDATE statistics
      SET
        count_total=0,
        count_price=0,
        sum_price=0,
        min_price=NULL,
        max_price=NULL
      WHERE id=1
      """)
    conexion.commit()
    cursor.close()
    conexion.close()
    print("Tablas limpiadas. Listo para re-ejecutar desde 4.3.")

# Descomentar para re-correr desde cero:
# limpiar_tablas()

In [165]:
cursor.close()
conexion.close()